 # 02 · Publicación del Catálogo de Reglas Trust & Safety — YelpConnect

 **Entregable:** E6 · Gobierno / Trust & Safety  
 **Tipo:** Publicación de catálogo gobernado  
 **Dominio:** Trust & Safety  
 **Schema:** `workspace.yelp_gov`  
 **Ejecución:** Idempotente mediante `MERGE`

 ---

 ## Propósito

 Publicar y mantener de forma gobernada las reglas heurísticas
 utilizadas por YelpConnect para identificar patrones
 potencialmente sospechosos en Reviews.

 Las estructuras físicas deben existir previamente:

 - `workspace.yelp_gov.evaluation_rule`
 - `workspace.yelp_gov.evaluation_rule_version`

 Estas tablas son creadas por:

 `00_setup/02_configurar_modelo_gobierno`

 ---

 ## Principio Trust & Safety

 Las reglas:

 - identifican **señales heurísticas**;
 - permiten priorizar casos para investigación;
 - no representan fraude confirmado;
 - no representan una probabilidad estadística de fraude.

 ---

 ## Modelo de gobierno

 ```text
 evaluation_rule
        1
        │
        │
        N
 evaluation_rule_version
 ```

 `evaluation_rule`

 **Grano:** 1 fila = 1 regla T&S.

 `evaluation_rule_version`

 **Grano:** 1 fila = 1 versión de una regla.

 ---

 ## Estado actual

 | Regla | Estado | Peso |
 |---|---|---:|
 | R001 | ACTIVE | 0.20 |
 | R002 | ACTIVE | 0.15 |
 | R003 | ACTIVE | 0.20 |
 | R004 | ACTIVE | 0.15 |
 | R005 | ACTIVE | 0.15 |
 | R006 | ACTIVE | 0.15 |
 | R007 | DRAFT | 0.00 |

 Las reglas ACTIVE suman:

 **1.00**

 R007 permanece experimental y no participa todavía en el
 `weighted_risk_score`.

 ---

 ## Este notebook SÍ

 - valida las tablas del catálogo;
 - publica R001-R007;
 - publica sus versiones;
 - mantiene `created_at` y `valid_from` históricos;
 - valida integridad regla ↔ versión;
 - valida reglas activas;
 - valida pesos.

 ## Este notebook NO

 - crea schemas;
 - crea tablas;
 - calcula features;
 - evalúa Reviews;
 - genera señales;
 - calcula Risk Score;
 - asigna P1/P2/P3;
 - modifica Owners o permisos.



In [0]:
# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F

# COMMAND ----------

In [0]:
# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"
GOV_SCHEMA = "yelp_gov"

GOV = f"{CATALOG}.{GOV_SCHEMA}"

RULE_TABLE = (
    f"{GOV}.evaluation_rule"
)

RULE_VERSION_TABLE = (
    f"{GOV}.evaluation_rule_version"
)

PIPELINE_NAME = (
    "E6_02_publicar_catalogo_reglas_ts"
)

EXPECTED_RULE_COUNT = 7
EXPECTED_ACTIVE_RULE_COUNT = 6
EXPECTED_DRAFT_VERSION_COUNT = 1


print("=" * 72)
print("YELPCONNECT - CATÁLOGO TRUST & SAFETY")
print("=" * 72)

print(
    f"Rule table         : {RULE_TABLE}"
)

print(
    f"Rule version table : {RULE_VERSION_TABLE}"
)

print("=" * 72)

# COMMAND ----------


 ## 1 · Precondiciones

 Deben existir previamente:

 - `workspace.yelp_gov.evaluation_rule`
 - `workspace.yelp_gov.evaluation_rule_version`

 Si alguna tabla no existe, debe ejecutarse primero:

 `00_setup/02_configurar_modelo_gobierno`

In [0]:
# COMMAND ----------

# ============================================================
# 3. VALIDAR TABLAS DE GOBIERNO
# ============================================================

required_tables = [
    RULE_TABLE,
    RULE_VERSION_TABLE
]


missing_tables = [
    table_name
    for table_name in required_tables
    if not spark.catalog.tableExists(
        table_name
    )
]


if missing_tables:

    print(
        "❌ TABLAS DE GOBIERNO FALTANTES:"
    )

    for table_name in missing_tables:

        print(
            f" - {table_name}"
        )

    raise Exception(
        """
        Ejecutar primero:
        00_setup/02_configurar_modelo_gobierno
        """
    )


print(
    "✔ Tablas de gobierno disponibles."
)

In [0]:
# COMMAND ----------

# ============================================================
# 4. VALIDAR CONTRATO evaluation_rule
# ============================================================

required_rule_columns = {
    "rule_id",
    "rule_name",
    "rule_type",
    "business_definition",
    "owner",
    "active_flag",
    "created_at"
}


actual_rule_columns = {
    field.name
    for field in spark.table(
        RULE_TABLE
    ).schema.fields
}


missing_rule_columns = sorted(
    required_rule_columns
    -
    actual_rule_columns
)


if missing_rule_columns:

    raise Exception(
        "❌ evaluation_rule no cumple el contrato. "
        "Faltan columnas: "
        +
        ", ".join(
            missing_rule_columns
        )
    )


print(
    "✔ Contrato evaluation_rule validado."
)

In [0]:

# COMMAND ----------

# ============================================================
# 5. VALIDAR CONTRATO evaluation_rule_version
# ============================================================

required_version_columns = {
    "rule_version_id",
    "rule_id",
    "version",
    "description",
    "parameters_json",
    "threshold",
    "weight",
    "valid_from",
    "status"
}


actual_version_columns = {
    field.name
    for field in spark.table(
        RULE_VERSION_TABLE
    ).schema.fields
}


missing_version_columns = sorted(
    required_version_columns
    -
    actual_version_columns
)


if missing_version_columns:

    raise Exception(
        "❌ evaluation_rule_version no cumple el contrato. "
        "Faltan columnas: "
        +
        ", ".join(
            missing_version_columns
        )
    )


print(
    "✔ Contrato evaluation_rule_version validado."
)

# COMMAND ----------

# Catálogo maestro de reglas

 ## 2 · `evaluation_rule`

 Esta tabla mantiene la identidad funcional de las reglas.

 ### Grano

 **1 fila = 1 regla Trust & Safety**

 El campo `owner` representa el responsable funcional.

 No debe confundirse con el `OWNER` técnico de Unity Catalog.


In [0]:
# COMMAND ----------

# ============================================================
# 6. DEFINICIÓN DE REGLAS
# ============================================================

rules = [

    # --------------------------------------------------------
    # R001
    # --------------------------------------------------------

    (
        "R001",
        "Burst Activity",
        "behavioral",
        "Detecta una concentración elevada de Reviews "
        "del mismo usuario dentro de una ventana temporal corta.",
        "Trust & Safety",
        True
    ),

    # --------------------------------------------------------
    # R002
    # --------------------------------------------------------

    (
        "R002",
        "Rating Deviation",
        "behavioral_content",
        "Detecta Reviews cuyo rating presenta una desviación "
        "elevada respecto al rating agregado del Business.",
        "Trust & Safety",
        True
    ),

    # --------------------------------------------------------
    # R003
    # --------------------------------------------------------

    (
        "R003",
        "Repeated Text",
        "content",
        "Detecta reutilización exacta de texto normalizado "
        "en múltiples Reviews.",
        "Trust & Safety",
        True
    ),

    # --------------------------------------------------------
    # R004
    # --------------------------------------------------------

    (
        "R004",
        "New Account High Activity",
        "behavioral",
        "Detecta actividad elevada durante los primeros días "
        "de vida de una cuenta.",
        "Trust & Safety",
        True
    ),

    # --------------------------------------------------------
    # R005
    # --------------------------------------------------------

    (
        "R005",
        "User-Business Concentration",
        "behavioral",
        "Detecta una concentración elevada de Reviews "
        "de un usuario hacia un mismo Business.",
        "Trust & Safety",
        True
    ),

    # --------------------------------------------------------
    # R006
    # --------------------------------------------------------

    (
        "R006",
        "Business Review Spike",
        "temporal",
        "Detecta incrementos anómalos del volumen diario "
        "de Reviews asociado a un Business.",
        "Trust & Safety",
        True
    ),

    # --------------------------------------------------------
    # R007 - EXPERIMENTAL
    # --------------------------------------------------------

    (
        "R007",
        "Cross-Business Repeated Review Pattern",
        "behavioral_content",
        "Detecta usuarios que reutilizan el mismo texto "
        "normalizado en Reviews dirigidas a múltiples "
        "Businesses. La regla se mantiene experimental "
        "hasta validar su precisión y volumen de activación.",
        "Trust & Safety",
        False
    )
]

# COMMAND ----------

In [0]:
# ============================================================
# 7. DATAFRAME DE REGLAS
# ============================================================

rule_schema = """
rule_id STRING,
rule_name STRING,
rule_type STRING,
business_definition STRING,
owner STRING,
active_flag BOOLEAN
"""


df_rules = (

    spark

    .createDataFrame(
        rules,
        rule_schema
    )

    .withColumn(
        "created_at",
        F.current_timestamp()
    )
)


defined_rule_count = (
    df_rules.count()
)


if defined_rule_count != EXPECTED_RULE_COUNT:

    raise Exception(
        f"""
        ❌ Cantidad inesperada de reglas.

        Esperadas:
        {EXPECTED_RULE_COUNT}

        Definidas:
        {defined_rule_count}
        """
    )


print(
    f"✔ Reglas definidas: "
    f"{defined_rule_count}"
)


display(
    df_rules
    .orderBy(
        "rule_id"
    )
)

# COMMAND ----------

In [0]:

# ============================================================
# 8. VALIDAR rule_id ÚNICO
# ============================================================

duplicate_rule_ids = (

    df_rules

    .groupBy(
        "rule_id"
    )

    .count()

    .filter(
        "count > 1"
    )

    .count()
)


if duplicate_rule_ids > 0:

    raise Exception(
        f"""
        ❌ Existen {duplicate_rule_ids}
        rule_id duplicados.
        """
    )


print(
    "✔ rule_id únicos."
)

# COMMAND ----------


 

## 3 · Publicación del catálogo maestro

 Se utiliza `MERGE` por:

 `rule_id`

 El campo `created_at` se mantiene para reglas existentes.

 Esto permite preservar la fecha original de incorporación de la regla.

In [0]:

# COMMAND ----------

# ============================================================
# 9. MERGE evaluation_rule
# ============================================================

df_rules.createOrReplaceTempView(
    "src_evaluation_rule"
)


spark.sql(
    f"""
    MERGE INTO
        {RULE_TABLE} AS target

    USING
        src_evaluation_rule AS source

    ON
        target.rule_id =
        source.rule_id

    WHEN MATCHED THEN

        UPDATE SET

            target.rule_name =
                source.rule_name,

            target.rule_type =
                source.rule_type,

            target.business_definition =
                source.business_definition,

            target.owner =
                source.owner,

            target.active_flag =
                source.active_flag

    WHEN NOT MATCHED THEN

        INSERT
        (
            rule_id,
            rule_name,
            rule_type,
            business_definition,
            owner,
            active_flag,
            created_at
        )

        VALUES
        (
            source.rule_id,
            source.rule_name,
            source.rule_type,
            source.business_definition,
            source.owner,
            source.active_flag,
            source.created_at
        )
    """
)


print(
    "✔ Catálogo maestro actualizado."
)

# Versionado de reglas

 ## 4 · `evaluation_rule_version`

 Cada versión conserva:

 - configuración;
 - parámetros;
 - threshold;
 - peso;
 - estado;
 - fecha inicial de vigencia.

 ### Grano

 **1 fila = 1 versión de una regla**

 ---

 ### R007

 R007 se mantiene como:

 - versión `0.1`;
 - estado `DRAFT`;
 - peso `0.0`;
 - `active_flag = false`.

 Esto permite conservarla dentro del catálogo sin alterar todavía
 el Risk Score.



In [0]:

# ============================================================
# 10. DEFINICIÓN DE VERSIONES
# ============================================================

rule_versions = [

    # --------------------------------------------------------
    # R001
    # --------------------------------------------------------

    (
        "R001-v1.0",
        "R001",
        "1.0",
        "8 o más Reviews del mismo usuario "
        "en una ventana trailing de 30 minutos.",
        """
        {
            "window_minutes": 30,
            "min_reviews": 8
        }
        """,
        8.0,
        0.20,
        "ACTIVE"
    ),

    # --------------------------------------------------------
    # R002
    # --------------------------------------------------------

    (
        "R002-v1.0",
        "R002",
        "1.0",
        "Desviación absoluta de al menos 3 estrellas "
        "respecto al rating agregado del Business.",
        """
        {
            "rating_difference": 3.0
        }
        """,
        3.0,
        0.15,
        "ACTIVE"
    ),

    # --------------------------------------------------------
    # R003
    # --------------------------------------------------------

    (
        "R003-v1.0",
        "R003",
        "1.0",
        "Texto normalizado de al menos 40 caracteres "
        "repetido al menos 3 veces.",
        """
        {
            "min_repetitions": 3,
            "min_text_length": 40
        }
        """,
        3.0,
        0.20,
        "ACTIVE"
    ),

    # --------------------------------------------------------
    # R004
    # --------------------------------------------------------

    (
        "R004-v1.0",
        "R004",
        "1.0",
        "Cuenta con antigüedad máxima de 30 días "
        "y al menos 5 Reviews dentro del dataset.",
        """
        {
            "account_age_days": 30,
            "min_reviews": 5
        }
        """,
        5.0,
        0.15,
        "ACTIVE"
    ),

    # --------------------------------------------------------
    # R005
    # --------------------------------------------------------

    (
        "R005-v1.0",
        "R005",
        "1.0",
        "Al menos 80 por ciento de las Reviews del usuario "
        "concentradas en un mismo Business, con mínimo "
        "3 Reviews hacia dicho Business.",
        """
        {
            "concentration": 0.80,
            "min_business_reviews": 3
        }
        """,
        0.80,
        0.15,
        "ACTIVE"
    ),

    # --------------------------------------------------------
    # R006
    # --------------------------------------------------------

    (
        "R006-v1.0",
        "R006",
        "1.0",
        "Al menos 10 Reviews en un día y volumen "
        "igual o superior a 3 veces el promedio diario "
        "activo del Business.",
        """
        {
            "min_daily_reviews": 10,
            "spike_ratio": 3.0
        }
        """,
        3.0,
        0.15,
        "ACTIVE"
    ),

    # --------------------------------------------------------
    # R007 - EXPERIMENTAL
    # --------------------------------------------------------

    (
        "R007-v0.1",
        "R007",
        "0.1",
        "Usuario reutiliza el mismo texto normalizado "
        "en Reviews dirigidas a por lo menos "
        "2 Businesses diferentes.",
        """
        {
            "min_distinct_businesses": 2,
            "min_text_repetitions": 2,
            "experimental": true
        }
        """,
        2.0,
        0.0,
        "DRAFT"
    )
]

# COMMAND ----------

In [0]:


# ============================================================
# 11. DATAFRAME DE VERSIONES
# ============================================================

version_schema = """
rule_version_id STRING,
rule_id STRING,
version STRING,
description STRING,
parameters_json STRING,
threshold DOUBLE,
weight DOUBLE,
status STRING
"""


df_rule_versions = (

    spark

    .createDataFrame(
        rule_versions,
        version_schema
    )

    .withColumn(
        "parameters_json",

        F.regexp_replace(
            F.trim(
                F.col(
                    "parameters_json"
                )
            ),
            r"\s+",
            " "
        )
    )

    .withColumn(
        "valid_from",
        F.current_timestamp()
    )
)


defined_version_count = (
    df_rule_versions.count()
)


if defined_version_count != EXPECTED_RULE_COUNT:

    raise Exception(
        f"""
        ❌ Cantidad inesperada de versiones.

        Esperadas:
        {EXPECTED_RULE_COUNT}

        Definidas:
        {defined_version_count}
        """
    )


print(
    f"✔ Versiones definidas: "
    f"{defined_version_count}"
)


display(
    df_rule_versions

    .select(
        "rule_version_id",
        "rule_id",
        "version",
        "threshold",
        "weight",
        "status",
        "parameters_json"
    )

    .orderBy(
        "rule_id"
    )
)

# COMMAND ----------


In [0]:

# ============================================================
# 12. VALIDAR rule_version_id ÚNICO
# ============================================================

duplicate_version_ids = (

    df_rule_versions

    .groupBy(
        "rule_version_id"
    )

    .count()

    .filter(
        "count > 1"
    )

    .count()
)


if duplicate_version_ids > 0:

    raise Exception(
        f"""
        ❌ Existen {duplicate_version_ids}
        rule_version_id duplicados.
        """
    )


print(
    "✔ rule_version_id únicos."
)

# COMMAND ----------


## 5 · Publicación de versiones

 El `MERGE` utiliza:

 `rule_version_id`

 como clave estable.

 El campo:

 `valid_from`

 no se actualiza para versiones existentes.

 De esta forma se conserva la fecha original de registro de la versión.


In [0]:
 
# COMMAND ----------

# ============================================================
# 13. MERGE evaluation_rule_version
# ============================================================

df_rule_versions.createOrReplaceTempView(
    "src_evaluation_rule_version"
)


spark.sql(
    f"""
    MERGE INTO
        {RULE_VERSION_TABLE} AS target

    USING
        src_evaluation_rule_version AS source

    ON
        target.rule_version_id =
        source.rule_version_id

    WHEN MATCHED THEN

        UPDATE SET

            target.rule_id =
                source.rule_id,

            target.version =
                source.version,

            target.description =
                source.description,

            target.parameters_json =
                source.parameters_json,

            target.threshold =
                source.threshold,

            target.weight =
                source.weight,

            target.status =
                source.status

    WHEN NOT MATCHED THEN

        INSERT
        (
            rule_version_id,
            rule_id,
            version,
            description,
            parameters_json,
            threshold,
            weight,
            valid_from,
            status
        )

        VALUES
        (
            source.rule_version_id,
            source.rule_id,
            source.version,
            source.description,
            source.parameters_json,
            source.threshold,
            source.weight,
            source.valid_from,
            source.status
        )
    """
)


print(
    "✔ Versiones actualizadas."
)

# COMMAND ----------


 # Validaciones del catálogo

 ## 6 · Integridad regla ↔ versión

 Toda versión debe poseer una regla padre.

 Una versión huérfana representa un error de gobierno y debe detener
 la ejecución.


In [0]:

# COMMAND ----------

# ============================================================
# 14. VERSIONES SIN REGLA PADRE
# ============================================================

orphan_versions = (

    spark.table(
        RULE_VERSION_TABLE
    )
    .alias("v")

    .join(

        spark.table(
            RULE_TABLE
        )
        .alias("r"),

        F.col(
            "v.rule_id"
        )
        ==
        F.col(
            "r.rule_id"
        ),

        "left_anti"
    )

    .count()
)


if orphan_versions > 0:

    raise Exception(
        f"""
        ❌ Se encontraron {orphan_versions}
        versiones sin regla padre.
        """
    )


print(
    "✔ Todas las versiones tienen regla padre."
)

# COMMAND ----------


 ## 7 · Reglas activas

 Toda regla con:

 `active_flag = true`

 debe poseer al menos una versión:

 `status = ACTIVE`

 De lo contrario la regla no podría ejecutarse de forma reproducible.


In [0]:

# COMMAND ----------

# ============================================================
# 15. VALIDAR REGLAS ACTIVAS
# ============================================================

active_without_active_version = (

    spark.table(
        RULE_TABLE
    )
    .alias("r")

    .filter(
        F.col(
            "r.active_flag"
        )
        ==
        True
    )

    .join(

        spark.table(
            RULE_VERSION_TABLE
        )
        .alias("v")

        .filter(
            F.col(
                "v.status"
            )
            ==
            "ACTIVE"
        ),

        F.col(
            "r.rule_id"
        )
        ==
        F.col(
            "v.rule_id"
        ),

        "left"
    )

    .filter(
        F.col(
            "v.rule_version_id"
        ).isNull()
    )

    .count()
)


if active_without_active_version > 0:

    raise Exception(
        f"""
        ❌ Existen {active_without_active_version}
        reglas activas sin versión ACTIVE.
        """
    )


print(
    "✔ Reglas activas con versión vigente."
)

# COMMAND ----------

## 8 · Pesos del modelo

 Los pesos de las versiones `ACTIVE` participan posteriormente en:

 `weighted_risk_score`

 Actualmente:

 ```text
 R001 = 0.20
 R002 = 0.15
 R003 = 0.20
 R004 = 0.15
 R005 = 0.15
 R006 = 0.15
 ----------------
 TOTAL = 1.00
 ```

 R007:

 `weight = 0.0`

 porque permanece en estado `DRAFT`.

 **El peso no representa probabilidad de fraude.**

 Es un componente del score heurístico utilizado para priorización.


In [0]:
 
# COMMAND ----------

# ============================================================
# 16. VALIDAR PESOS ACTIVE
# ============================================================

active_weight = (

    spark.table(
        RULE_VERSION_TABLE
    )

    .filter(
        F.col(
            "status"
        )
        ==
        "ACTIVE"
    )

    .agg(

        F.round(
            F.sum(
                "weight"
            ),
            4
        )
        .alias(
            "total_weight"
        )
    )

    .first()
    .total_weight
)


print(
    f"Peso total reglas ACTIVE: "
    f"{active_weight}"
)


if active_weight is None:

    raise Exception(
        """
        ❌ No existen versiones ACTIVE.
        """
    )


if abs(
    active_weight - 1.0
) > 0.0001:

    raise Exception(
        f"""
        ❌ Los pesos de las reglas ACTIVE
        no suman 1.0.

        Peso actual:
        {active_weight}
        """
    )


print(
    "✔ Pesos ACTIVE suman 1.0."
)

# COMMAND ----------


In [0]:

# ============================================================
# 17. CONSTRUIR VISTA DEL CATÁLOGO
# ============================================================

catalog_view = (

    spark.table(
        RULE_TABLE
    )
    .alias("r")

    .join(

        spark.table(
            RULE_VERSION_TABLE
        )
        .alias("v"),

        F.col(
            "r.rule_id"
        )
        ==
        F.col(
            "v.rule_id"
        ),

        "left"
    )

    .select(

        F.col(
            "r.rule_id"
        ),

        F.col(
            "r.rule_name"
        ),

        F.col(
            "r.rule_type"
        ),

        F.col(
            "r.owner"
        ),

        F.col(
            "r.active_flag"
        ),

        F.col(
            "v.rule_version_id"
        ),

        F.col(
            "v.version"
        ),

        F.col(
            "v.threshold"
        ),

        F.col(
            "v.weight"
        ),

        F.col(
            "v.status"
        ),

        F.col(
            "v.valid_from"
        )
    )

    .orderBy(
        "rule_id",
        "version"
    )
)


display(
    catalog_view
)

# COMMAND ----------



 ## 9 · Reglas actualmente ejecutables

 Una regla se considera ejecutable cuando:

 ```text
 evaluation_rule.active_flag = true
 AND
 evaluation_rule_version.status = ACTIVE
 ```

 Actualmente deben resultar ejecutables:

 **R001-R006**

In [0]:

# COMMAND ----------

# ============================================================
# 18. REGLAS EJECUTABLES
# ============================================================

executable_rules = (

    catalog_view

    .filter(
        (
            F.col(
                "active_flag"
            )
            ==
            True
        )

        &

        (
            F.col(
                "status"
            )
            ==
            "ACTIVE"
        )
    )
)


display(
    executable_rules

    .select(
        "rule_id",
        "rule_name",
        "rule_version_id",
        "threshold",
        "weight"
    )
)


executable_rule_count = (
    executable_rules.count()
)


if (
    executable_rule_count
    !=
    EXPECTED_ACTIVE_RULE_COUNT
):

    raise Exception(
        f"""
        ❌ Cantidad inesperada de reglas ejecutables.

        Esperadas:
        {EXPECTED_ACTIVE_RULE_COUNT}

        Encontradas:
        {executable_rule_count}
        """
    )


print(
    f"✔ Reglas ejecutables: "
    f"{executable_rule_count}"
)


## 10 · Reglas experimentales

 Las versiones `DRAFT` se conservan dentro del catálogo para permitir
 evolución controlada.

 Actualmente:

 **R007-v0.1**

 permanece experimental y no participa en el Risk Score.

In [0]:

# COMMAND ----------

# ============================================================
# 19. REGLAS DRAFT
# ============================================================

draft_versions = (

    catalog_view

    .filter(
        F.col(
            "status"
        )
        ==
        "DRAFT"
    )
)


display(
    draft_versions

    .select(
        "rule_id",
        "rule_name",
        "rule_version_id",
        "threshold",
        "weight",
        "status",
        "active_flag"
    )
)


draft_version_count = (
    draft_versions.count()
)


if (
    draft_version_count
    !=
    EXPECTED_DRAFT_VERSION_COUNT
):

    raise Exception(
        f"""
        ❌ Cantidad inesperada de versiones DRAFT.

        Esperadas:
        {EXPECTED_DRAFT_VERSION_COUNT}

        Encontradas:
        {draft_version_count}
        """
    )


print(
    f"✔ Versiones DRAFT: "
    f"{draft_version_count}"
)

# COMMAND ----------

In [0]:

# ============================================================
# 20. RESUMEN
# ============================================================

rule_count = (
    spark.table(
        RULE_TABLE
    )
    .count()
)


active_rule_count = (

    spark.table(
        RULE_TABLE
    )

    .filter(
        F.col(
            "active_flag"
        )
        ==
        True
    )

    .count()
)


total_version_count = (
    spark.table(
        RULE_VERSION_TABLE
    )
    .count()
)


print()
print("=" * 72)
print("CATÁLOGO TRUST & SAFETY ACTUALIZADO")
print("=" * 72)

print(
    f"Reglas registradas : "
    f"{rule_count}"
)

print(
    f"Reglas activas     : "
    f"{active_rule_count}"
)

print(
    f"Versiones          : "
    f"{total_version_count}"
)

print(
    f"Versiones DRAFT    : "
    f"{draft_version_count}"
)

print(
    f"Peso reglas ACTIVE : "
    f"{active_weight}"
)

print()

print(
    "R007 permanece experimental "
    "y NO participa todavía en el Risk Score."
)

print("=" * 72)

# COMMAND ----------



 ## Definition of Done

 `02_publicar_catalogo_reglas_ts` se considera completado cuando:

 - [ ] `evaluation_rule` existe previamente;
 - [ ] `evaluation_rule_version` existe previamente;
 - [ ] R001-R006 están activas;
 - [ ] R001-R006 poseen versión `ACTIVE`;
 - [ ] R007 está registrada;
 - [ ] R007-v0.1 permanece `DRAFT`;
 - [ ] R007 tiene peso `0.0`;
 - [ ] todas las versiones poseen regla padre;
 - [ ] las reglas activas poseen versión activa;
 - [ ] existen 6 reglas ejecutables;
 - [ ] los pesos ACTIVE suman `1.0`;
 - [ ] no se evaluaron Reviews;
 - [ ] no se calculó Risk Score.

 ---

 ## Siguiente etapa

 Una vez publicados ambos catálogos:

 ```text
 01_publicar_catalogo_reglas_dq_silver
         ↓
 02_publicar_catalogo_reglas_ts
         ↓
 E7/01_ingestar_bronze
 ```

 La siguiente etapa abandona la configuración de gobierno e inicia
 el procesamiento de datos.


In [0]:
# COMMAND ----------

# ============================================================
# 21. RESUMEN FINAL
# ============================================================

print()
print("=" * 72)
print("PUBLICACIÓN T&S COMPLETADA")
print("=" * 72)

print()
print(f"Catálogo reglas   : {RULE_TABLE}")
print(f"Catálogo versiones: {RULE_VERSION_TABLE}")

print()
print(f"Reglas totales    : {rule_count}")
print(f"Reglas ejecutables: {executable_rule_count}")
print(f"Versiones DRAFT   : {draft_version_count}")
print(f"Peso ACTIVE       : {active_weight}")

print()
print("✔ Catálogo publicado mediante MERGE.")
print("✔ R001-R006 disponibles para evaluación.")
print("✔ R007 preservada como experimental.")
print("✔ No se evaluaron Reviews.")
print("✔ No se calculó Risk Score.")

print()
print("Siguiente paso:")
print("→ E7/01_ingestar_bronze")

print("=" * 72)